# Persistence Baseline and Time-Based Validation

This notebook evaluates the persistence rule on the original outer test folds and creates leakage-safe Full TA train, validation, and test folds for later hyperparameter tuning.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "models").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root

WindowsPath('d:/SET50_direction_prediction_paper')

## Create Time-Based Validation Folds

The final year of each outer training period becomes validation. The outer test year remains untouched.

In [2]:
from models.time_based_validation import create_full_ta_validation_folds, create_scaled_full_ta_validation_folds

tabular_validation_dir = create_full_ta_validation_folds()
lstm_validation_dir = create_scaled_full_ta_validation_folds()
tabular_validation_dir, lstm_validation_dir

(WindowsPath('D:/SET50_direction_prediction_paper/data-folds-full-ta-validation'),
 WindowsPath('D:/SET50_direction_prediction_paper/data-folds-full-ta-validation-nn'))

## Inspect Validation Fold Schedule

In [3]:
from dataclasses import asdict
import pandas as pd
from models.time_based_validation import discover_validation_folds

validation_schedule = pd.DataFrame(
    asdict(spec) for spec in discover_validation_folds(tabular_validation_dir)
)
validation_schedule

,fold,train_path,validation_path,test_path,train_start_year,train_end_year,validation_year,test_year
0,fold_1,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,2012,2020,2021,2022
1,fold_2,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,2012,2021,2022,2023
2,fold_3,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,2012,2022,2023,2024
3,fold_4,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,D:\SET50_direction_prediction_paper\data-folds...,2012,2023,2024,2025


## Run Persistence Baseline

Prediction rule: `Predicted Close(t+1) = Close(t)`.

In [4]:
from models.persistence_baseline import run_persistence_baseline

persistence_metrics = run_persistence_baseline()
persistence_metrics

persistence_current_close:   0%|          | 0/4 [00:00<?, ?fold/s]

     fold  train_start_year  train_end_year  test_year  n_train  n_test  \
0  fold_1              2012            2021       2022     2419     241   
1  fold_2              2012            2022       2023     2660     243   
2  fold_3              2012            2023       2024     2903     244   
3  fold_4              2012            2024       2025     3147     234   

       rmse       mae      mape        r2  direction_accuracy  
0  6.789210  5.168257  0.525856  0.906948            0.004149  
1  7.319332  5.522551  0.596424  0.972940            0.004115  
2  6.565448  4.870697  0.561655  0.977772            0.000000  
3  9.471055  7.330855  0.929988  0.965373            0.000000  
      train_start_year  train_end_year  test_year  n_train  n_test      rmse  \
mean            2012.0          2022.5     2023.5  2782.25   240.5  7.536262   

          mae      mape        r2  direction_accuracy  
mean  5.72309  0.653481  0.955758            0.002066  


,fold,train_start_year,train_end_year,test_year,n_train,n_test,rmse,mae,mape,r2,direction_accuracy
0,fold_1,2012,2021,2022,2419,241,6.789210,5.168257,0.525856,0.906948,0.004149
1,fold_2,2012,2022,2023,2660,243,7.319332,5.522551,0.596424,0.972940,0.004115
2,fold_3,2012,2023,2024,2903,244,6.565448,4.870697,0.561655,0.977772,0.000000
3,fold_4,2012,2024,2025,3147,234,9.471055,7.330855,0.929988,0.965373,0.000000
